# Portfolio Optimization

This notebook uses historical returns and predicted volatility to build and compare portfolio strategies.

The goal is to compare simple portfolio baselines against optimized portfolios. The strategies include an equal-weight portfolio, a historical minimum-volatility portfolio, a predictive minimum-volatility portfolio, and a maximum Sharpe ratio portfolio.

# Imports

In [39]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.optimize import minimize

import matplotlib.pyplot as plt

# Paths

In [40]:
INTEGRATED_DATA_PATH = Path("../../data/processed/integrated")
MODELING_PATH = Path("../../data/processed/modeling")
PORTFOLIO_OUTPUT_PATH = Path("../../data/processed/portfolio_optimization")

In [41]:
if not PORTFOLIO_OUTPUT_PATH.exists():
    PORTFOLIO_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory {PORTFOLIO_OUTPUT_PATH} already exists.") 

Directory ..\..\data\processed\portfolio_optimization already exists.


# Load Data

## Daily Market Data

This dataset gives the historical daily returns used to build the portfolio return matrix.

In [42]:
daily_market_data = pd.read_csv(
    INTEGRATED_DATA_PATH / "daily_market_data.csv",
    parse_dates=["Date"]
)

## Random Forest Predictions

For this notebook, we use the Random Forest test predictions as the predictive volatility input for portfolio optimization.

In [43]:
rf_predictions = pd.read_csv(
    MODELING_PATH / "random_forest" / "test_predictions.csv",
    parse_dates=["Date"]
)

## Validate Loaded Data

Before creating portfolios, we quickly check that both input files loaded correctly.

In [44]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,vix,treasury_10yr_pct,yield_curve_spread,is_inverted,fed_funds_rate_pct,unemployment_rate_pct,recession_flag,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-02,AAPL,40.267082,NaN,1.44,0.0144,9.77,2.46,1.02,0,1.41,4.0,0.0,248.859,0.004253,Apple Inc.,Information Technology,Stock
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,9.15,2.44,1.03,0,1.41,4.0,0.0,248.859,0.004253,Apple Inc.,Information Technology,Stock
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,9.22,2.46,1.05,0,1.41,4.0,0.0,248.859,0.004253,Apple Inc.,Information Technology,Stock
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,9.22,2.47,1.08,0,1.41,4.0,0.0,248.859,0.004253,Apple Inc.,Information Technology,Stock
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,9.52,2.49,1.04,0,1.41,4.0,0.0,248.859,0.004253,Apple Inc.,Information Technology,Stock


In [45]:
daily_market_data.shape

(42210, 18)

In [46]:
rf_predictions.head()

,Date,ticker,future_volatility_20d,predicted_future_volatility_20d
0,2024-01-02,MSFT,0.010424,0.012478
1,2024-01-02,PG,0.011748,0.011070
2,2024-01-02,CAT,0.015127,0.013427
3,2024-01-02,AGG,0.003369,0.003997
4,2024-01-02,KO,0.006132,0.009372


In [47]:
rf_predictions.shape

(10101, 4)

# Create Return Matrix

## Pivot Daily Returns

This turns the long daily market dataset into a matrix with dates as rows and tickers as columns.

In [48]:
returns_matrix = daily_market_data.pivot(
    index="Date",
    columns="ticker",
    values="daily_return"
).sort_index()

## Remove Missing Returns

Rows with missing return values are removed so every portfolio calculation uses a complete set of asset returns.

In [49]:
returns_matrix = returns_matrix.dropna()

## Check Return Matrix

These checks confirm the return matrix has the expected shape and ticker columns.

In [50]:
returns_matrix.head()

ticker,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,...,PG,QQQ,SPY,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-03,-0.000174,0.000092,0.012775,0.001528,-0.002637,0.001019,-0.002196,0.005432,0.008381,0.004654,...,-0.001213,0.009717,0.006325,0.004781,0.010490,0.009955,-0.002903,0.006971,-0.020549,0.019640
2018-01-04,0.004645,-0.000641,0.004476,0.013734,0.005127,0.014325,0.014084,0.004463,0.017154,0.008801,...,0.007068,0.001750,0.004215,-0.000159,0.004340,0.003718,-0.017227,0.008307,0.003243,0.001384
2018-01-05,0.011385,-0.000641,0.016163,0.015805,-0.001036,-0.006419,-0.000217,0.012278,0.009060,0.012398,...,0.000658,0.010043,0.006664,-0.002855,0.019069,0.023949,0.000494,0.006694,-0.002281,-0.000806
2018-01-08,-0.003714,-0.000276,0.014425,0.025129,-0.000160,0.001477,-0.001519,-0.005083,-0.004611,0.001020,...,0.005261,0.003891,0.001829,-0.000637,-0.017357,0.004038,0.005182,-0.000512,-0.001715,0.004496
2018-01-09,-0.000114,-0.002752,0.004676,0.002409,-0.004628,0.005069,0.004999,-0.000813,0.007162,-0.000680,...,-0.007305,0.000061,0.002264,-0.013373,0.004983,-0.001927,-0.012888,0.000853,-0.003668,-0.004246


In [51]:
returns_matrix.shape

(2009, 21)

# Create Portfolio Helper Functions

## Annualization Setting

We use 252 trading days as the standard yearly trading-day assumption.

In [52]:
TRADING_DAYS = 252

## Portfolio Return

This function calculates the annualized return for a portfolio based on asset weights and average daily returns.

In [53]:
def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns) * TRADING_DAYS

## Portfolio Volatility

This function calculates annualized portfolio volatility using the covariance matrix.

In [54]:
def portfolio_volatility(weights, covariance_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(covariance_matrix, weights))) * np.sqrt(TRADING_DAYS)

## Sharpe Ratio

This function measures risk-adjusted return by comparing portfolio return against the risk-free rate.

In [55]:
def portfolio_sharpe(weights, mean_returns, covariance_matrix, risk_free_rate=0.0):
    port_return = portfolio_return(weights, mean_returns)
    port_vol = portfolio_volatility(weights, covariance_matrix)

    if port_vol == 0:
        return 0

    return (port_return - risk_free_rate) / port_vol

# Create Portfolio Evaluation Functions

## Maximum Drawdown

Maximum drawdown measures the largest peak-to-trough loss during the evaluation period.

In [56]:
def max_drawdown(portfolio_returns):
    cumulative = (1 + portfolio_returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max

    return drawdown.min()

## Portfolio Evaluation Metrics

This function summarizes each strategy using return, volatility, Sharpe ratio, drawdown, and cumulative return.

In [57]:
def evaluate_portfolio(strategy_name, weights, returns_data, risk_free_rate=0.0):
    daily_portfolio_returns = returns_data.dot(weights)

    annual_return = daily_portfolio_returns.mean() * TRADING_DAYS
    annual_volatility = daily_portfolio_returns.std() * np.sqrt(TRADING_DAYS)

    if annual_volatility == 0:
        sharpe_ratio = 0
    else:
        sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility

    cumulative_return = (1 + daily_portfolio_returns).prod() - 1
    drawdown = max_drawdown(daily_portfolio_returns)

    return {
        "strategy": strategy_name,
        "annualized_return": annual_return,
        "annualized_volatility": annual_volatility,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": drawdown,
        "cumulative_return": cumulative_return
    }

# Portfolio Setup

For portfolio optimization, SPY is treated as the benchmark instead of being included as one of the optimized portfolio holdings.

The optimized portfolios use the remaining assets. We train portfolio weights using data before 2024 and evaluate performance on 2024 and later.

## Benchmark Asset

SPY is saved as the benchmark so we can compare optimized portfolios against the broader U.S. market.

In [58]:
BENCHMARK = "SPY"

## Optimized Asset Universe

The optimized portfolio uses all assets except SPY.

In [59]:
portfolio_assets = [ticker for ticker in returns_matrix.columns if ticker != BENCHMARK] # removes SPY

## Portfolio and Benchmark Returns

This separates the asset return matrix used for optimization from the SPY benchmark return series.

In [60]:
portfolio_returns = returns_matrix[portfolio_assets]
benchmark_returns = returns_matrix[BENCHMARK]

# Train/Test Split

The portfolio weights are built using data before 2024 and evaluated on data from 2024 onward.

In [61]:
train_returns = portfolio_returns[portfolio_returns.index < "2024-01-01"]
test_returns = portfolio_returns[portfolio_returns.index >= "2024-01-01"]

## Benchmark Test Period

The SPY benchmark is aligned to the same test dates as the optimized portfolios.

In [62]:
test_benchmark_returns = benchmark_returns.loc[test_returns.index]

## Confirm Portfolio Assets

This confirms which tickers are included in the optimized portfolio universe.

In [63]:
portfolio_assets

['AAPL',
 'AGG',
 'AMZN',
 'CAT',
 'GLD',
 'JPM',
 'KO',
 'LLY',
 'LMT',
 'MSFT',
 'NEE',
 'PG',
 'QQQ',
 'TLT',
 'UNH',
 'V',
 'VNQ',
 'VXUS',
 'VZ',
 'XOM']

# Estimate Historical Return and Risk

We use the training period to estimate each asset's average return, covariance, and correlation. This avoids using future return data when creating the portfolio weights.

## Mean Returns, Covariance, and Correlation

These historical estimates are the main inputs for the historical optimization strategies.

In [64]:
mean_returns = train_returns.mean()
covariance_matrix = train_returns.cov()
correlation_matrix = train_returns.corr()

## Check Historical Return Estimates

This displays a small sample of the average daily returns from the training period.

In [65]:
mean_returns.head()

ticker
AAPL    0.001230
AGG     0.000044
AMZN    0.000870
CAT     0.000727
GLD     0.000321
dtype: float64

# Risk-Free Rate

The risk-free rate comes from the FRED 3-month Treasury yield. We use the training average when optimizing portfolios and the test average when evaluating portfolio Sharpe ratios.

## Daily Risk-Free Series

This creates one daily risk-free-rate value per date.

In [66]:
daily_risk_free = (
    daily_market_data[["Date", "risk_free_rate_decimal"]]
    .drop_duplicates("Date")
    .set_index("Date")
    .sort_index()
)

## Train and Test Risk-Free Rates

The training average is used during optimization, and the test average is used during portfolio evaluation.

In [67]:
train_risk_free_rate = daily_risk_free.loc[train_returns.index, "risk_free_rate_decimal"].mean()
test_risk_free_rate = daily_risk_free.loc[test_returns.index, "risk_free_rate_decimal"].mean()

## Check Risk-Free Rates

This confirms the risk-free-rate values that will be used in optimization and evaluation.

In [68]:
train_risk_free_rate, test_risk_free_rate

(np.float64(0.019715716180371348), np.float64(0.04694930139720557))

# Optimization Constraints

All portfolios are long-only, which means no short selling. Each portfolio must add up to 100%, and no single asset can be more than 25% of the portfolio. The cap keeps the optimized portfolios more diversified.

NOTE: keeping it simple for now

## Asset Count

This counts how many assets are included in the optimized portfolio universe.

In [69]:
n_assets = len(portfolio_assets)

n_assets

20

## Maximum Asset Weight

No single asset can receive more than 25% of the portfolio.

In [70]:
MAX_WEIGHT = 0.25

## Initial Equal Weights

The optimizer starts from an equal-weight portfolio before searching for better weights.

In [71]:
initial_weights = np.repeat(1 / n_assets, n_assets)

## Bounds

Bounds keep every asset weight between 0% and the maximum asset weight.

In [72]:
bounds = tuple((0, MAX_WEIGHT) for _ in range(n_assets))

## Sum-to-One Constraint

This constraint forces the final portfolio weights to add up to 100%.

In [73]:
constraints = {
    "type": "eq",
    "fun": lambda weights: np.sum(weights) - 1
}

# Build Historical Portfolio Strategies

Now we will create the first three portfolio strategies:

1. Equal Weight: Every asset gets the same weight.
2. Historical Minimum Volatility: Choose weights that minimize portfolio risk using historical covariance.
3. Historical Maximum Sharpe: Choose weights that maximize risk-adjusted return using historical returns and risk.

## Equal Weight Portfolio

The equal-weight portfolio is the simple baseline strategy. Every asset receives the same portfolio weight.

In [74]:
equal_weight_weights = initial_weights

## Historical Minimum Volatility Portfolio

This strategy uses historical covariance to find the portfolio with the lowest estimated volatility.

In [75]:
historical_min_vol_result = minimize(
    lambda weights: portfolio_volatility(weights, covariance_matrix),
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

In [76]:
historical_min_vol_result.success, historical_min_vol_result.message

(True, 'Optimization terminated successfully')

In [77]:
historical_min_vol_weights = historical_min_vol_result.x

In [78]:
pd.Series(
    historical_min_vol_weights,
    index=portfolio_assets
).sort_values(ascending=False)

AGG     2.500000e-01
GLD     2.325487e-01
TLT     2.230235e-01
VZ      7.256858e-02
PG      4.329584e-02
JPM     4.245400e-02
LMT     4.166495e-02
LLY     2.293884e-02
KO      2.103493e-02
VXUS    2.078724e-02
CAT     1.475208e-02
XOM     1.431423e-02
UNH     6.171076e-04
QQQ     7.133453e-18
V       6.799579e-18
AAPL    0.000000e+00
AMZN    0.000000e+00
MSFT    0.000000e+00
NEE     0.000000e+00
VNQ     0.000000e+00
dtype: float64

## Historical Maximum Sharpe Portfolio

This strategy uses historical returns, historical covariance, and the risk-free rate to maximize risk-adjusted return.

In [81]:
historical_max_sharpe_result = minimize(
    lambda weights: -portfolio_sharpe(
        weights,
        mean_returns,
        covariance_matrix,
        train_risk_free_rate
    ),
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

In [82]:
historical_max_sharpe_result.success, historical_max_sharpe_result.message

(True, 'Optimization terminated successfully')

In [84]:
historical_max_sharpe_weights = historical_max_sharpe_result.x

pd.Series(
    historical_max_sharpe_weights,
    index=portfolio_assets
).sort_values(ascending=False)

LLY     2.500000e-01
GLD     2.500000e-01
MSFT    1.535486e-01
AAPL    1.499674e-01
TLT     1.037454e-01
CAT     4.362546e-02
AGG     4.283730e-02
UNH     6.275785e-03
XOM     1.540622e-16
JPM     1.153769e-16
LMT     8.571273e-17
AMZN    0.000000e+00
PG      0.000000e+00
NEE     0.000000e+00
KO      0.000000e+00
QQQ     0.000000e+00
V       0.000000e+00
VNQ     0.000000e+00
VXUS    0.000000e+00
VZ      0.000000e+00
dtype: float64

# Build Random Forest Predictive Portfolio Strategies

Now we use the Random Forest model's predicted future volatility as an input to portfolio optimization.

The Random Forest model predicts each asset's future 20-day volatility. To turn those predictions into a portfolio risk estimate, we combine the predicted volatility with the historical correlation matrix.

## Prepare Random Forest Predicted Volatility

In [86]:
rf_predicted_vol_matrix = rf_predictions.pivot(
    index="Date",
    columns="ticker",
    values="predicted_future_volatility_20d"
).sort_index()

In [88]:
rf_predicted_vol_matrix = rf_predicted_vol_matrix.reindex(columns=portfolio_assets)

In [89]:
rf_predicted_vol_matrix.head()

ticker,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2024-01-02,0.013355,0.003997,0.014693,0.013427,0.009116,0.012446,0.009372,0.013700,0.010232,0.012478,0.013467,0.011070,0.009441,0.009427,0.012493,0.009327,0.011686,0.008031,0.012752,0.013837
2024-01-03,0.013307,0.003997,0.013286,0.015306,0.008552,0.012431,0.009436,0.013574,0.010345,0.012341,0.013447,0.011108,0.009728,0.009405,0.012493,0.009095,0.012189,0.008078,0.012715,0.013309
2024-01-04,0.012590,0.003915,0.018122,0.015197,0.008469,0.012446,0.009354,0.013574,0.010156,0.011580,0.013836,0.009152,0.009838,0.009765,0.012493,0.009374,0.012430,0.008078,0.012648,0.013943
2024-01-05,0.012590,0.003920,0.017660,0.014116,0.008302,0.012446,0.009354,0.013595,0.009563,0.011068,0.013299,0.009215,0.009838,0.010000,0.012555,0.009346,0.013015,0.008078,0.013261,0.012839
2024-01-08,0.014510,0.003997,0.014696,0.013873,0.008627,0.012431,0.009264,0.013853,0.009493,0.012447,0.013279,0.009197,0.009441,0.009765,0.012555,0.009648,0.012082,0.008045,0.013269,0.014084


In [90]:
missing_assets = rf_predicted_vol_matrix.columns[
    rf_predicted_vol_matrix.isna().all()
].tolist()

missing_assets

[]

## Choose Prediction Date

For this simple version so far, we use the first available Random Forest prediction date in the test period. This avoids using the full test period to choose portfolio weights.

In [92]:
prediction_date = rf_predicted_vol_matrix.dropna().index.min()

In [94]:
rf_predicted_vol_by_asset = rf_predicted_vol_matrix.loc[prediction_date]

In [95]:
prediction_date

Timestamp('2024-01-02 00:00:00')

In [96]:
rf_predicted_vol_by_asset.sort_values(ascending=False)

ticker
AMZN    0.014693
XOM     0.013837
LLY     0.013700
NEE     0.013467
CAT     0.013427
AAPL    0.013355
VZ      0.012752
UNH     0.012493
MSFT    0.012478
JPM     0.012446
VNQ     0.011686
PG      0.011070
LMT     0.010232
QQQ     0.009441
TLT     0.009427
KO      0.009372
V       0.009327
GLD     0.009116
VXUS    0.008031
AGG     0.003997
Name: 2024-01-02 00:00:00, dtype: float64

## Create Predictive Covariance Matrix

The Random Forest gives predicted volatility for each asset, but portfolio optimization also needs covariance between assets.

To estimate predictive covariance, we will combine:

- Random Forest predicted volatility
- historical asset correlation

In [97]:
rf_predicted_covariance_matrix = pd.DataFrame(
    np.outer(rf_predicted_vol_by_asset, rf_predicted_vol_by_asset) * correlation_matrix.values,
    index=portfolio_assets,
    columns=portfolio_assets
)

In [98]:
rf_predicted_covariance_matrix.head()

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
AAPL,0.000178,0.000006,0.000122,0.000074,0.000009,7.653275e-05,0.000051,0.000062,4.790102e-05,0.000125,0.000071,0.000061,0.000108,-0.000017,0.000077,0.000078,0.000083,0.000069,0.000047,0.000060
AGG,0.000006,0.000016,0.000009,-0.000002,0.000014,-9.289694e-07,0.000004,0.000002,-5.846728e-07,0.000006,0.000013,0.000004,0.000006,0.000031,0.000002,0.000003,0.000012,0.000005,0.000004,-0.000002
AMZN,0.000122,0.000009,0.000216,0.000060,0.000012,5.615856e-05,0.000033,0.000048,3.120954e-05,0.000128,0.000054,0.000040,0.000110,-0.000004,0.000055,0.000067,0.000067,0.000063,0.000034,0.000039
CAT,0.000074,-0.000002,0.000060,0.000180,0.000003,1.062791e-04,0.000053,0.000046,5.758476e-05,0.000068,0.000049,0.000043,0.000064,-0.000035,0.000065,0.000065,0.000077,0.000069,0.000052,0.000111
GLD,0.000009,0.000014,0.000012,0.000003,0.000083,-6.162494e-06,0.000008,0.000004,7.655504e-06,0.000009,0.000024,0.000009,0.000009,0.000026,0.000006,0.000003,0.000015,0.000015,0.000009,0.000007


## Random Forest Predictive Minimum Volatility Portfolio

This portfolio uses Random Forest predicted volatility and historical correlation to choose the lowest-risk portfolio.

In [99]:
rf_predictive_min_vol_result = minimize(
    lambda weights: portfolio_volatility(weights, rf_predicted_covariance_matrix),
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

In [100]:
rf_predictive_min_vol_result.success, rf_predictive_min_vol_result.message

(True, 'Optimization terminated successfully')

In [103]:
rf_predictive_min_vol_weights = rf_predictive_min_vol_result.x

In [104]:
pd.Series(
    rf_predictive_min_vol_weights,
    index=portfolio_assets
).sort_values(ascending=False)

AGG     2.500000e-01
TLT     2.117377e-01
GLD     1.178222e-01
LMT     1.030125e-01
VXUS    7.586378e-02
V       7.200906e-02
KO      4.842229e-02
JPM     3.774558e-02
LLY     3.127592e-02
XOM     1.600526e-02
VZ      1.407628e-02
CAT     1.399528e-02
PG      8.034166e-03
QQQ     2.631055e-17
AMZN    1.527540e-17
UNH     2.019327e-18
NEE     4.136811e-19
AAPL    0.000000e+00
MSFT    0.000000e+00
VNQ     0.000000e+00
dtype: float64

## Random Forest Predictive Maximum Sharpe Portfolio

This portfolio uses historical expected returns, Random Forest predicted volatility, historical correlation, and the risk-free rate to maximize risk-adjusted return.

In [105]:
rf_predictive_max_sharpe_result = minimize(
    lambda weights: -portfolio_sharpe(
        weights,
        mean_returns,
        rf_predicted_covariance_matrix,
        train_risk_free_rate
    ),
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

In [106]:
rf_predictive_max_sharpe_result.success, rf_predictive_max_sharpe_result.message

(True, 'Optimization terminated successfully')

In [107]:
rf_predictive_max_sharpe_weights = rf_predictive_max_sharpe_result.x

In [108]:
pd.Series(
    rf_predictive_max_sharpe_weights,
    index=portfolio_assets
).sort_values(ascending=False)

LLY     2.500000e-01
MSFT    2.195523e-01
GLD     1.813171e-01
AAPL    1.734510e-01
TLT     8.138060e-02
CAT     4.859363e-02
V       4.570540e-02
VNQ     7.925176e-16
NEE     2.787809e-16
VZ      2.783830e-16
AMZN    1.593059e-16
KO      1.541518e-16
QQQ     1.354867e-16
VXUS    3.641530e-17
AGG     0.000000e+00
JPM     0.000000e+00
PG      0.000000e+00
LMT     0.000000e+00
UNH     0.000000e+00
XOM     0.000000e+00
dtype: float64

# Evaluate Portfolio Strategies

Now we compare all portfolio strategies on the test period. The weights were created using training data, and performance is evaluated on 2024 and later.

## Combine Strategy Weights

In [109]:
strategy_weights = pd.DataFrame(
    {
        "Equal Weight": equal_weight_weights,
        "Historical Min Vol": historical_min_vol_weights,
        "Historical Max Sharpe": historical_max_sharpe_weights,
        "RF Predictive Min Vol": rf_predictive_min_vol_weights,
        "RF Predictive Max Sharpe": rf_predictive_max_sharpe_weights
    },
    index=portfolio_assets
)

In [110]:
strategy_weights

,Equal Weight,Historical Min Vol,Historical Max Sharpe,RF Predictive Min Vol,RF Predictive Max Sharpe
AAPL,0.05,0.000000e+00,1.499674e-01,0.000000e+00,1.734510e-01
AGG,0.05,2.500000e-01,4.283730e-02,2.500000e-01,0.000000e+00
AMZN,0.05,0.000000e+00,0.000000e+00,1.527540e-17,1.593059e-16
CAT,0.05,1.475208e-02,4.362546e-02,1.399528e-02,4.859363e-02
GLD,0.05,2.325487e-01,2.500000e-01,1.178222e-01,1.813171e-01
JPM,0.05,4.245400e-02,1.153769e-16,3.774558e-02,0.000000e+00
KO,0.05,2.103493e-02,0.000000e+00,4.842229e-02,1.541518e-16
LLY,0.05,2.293884e-02,2.500000e-01,3.127592e-02,2.500000e-01
LMT,0.05,4.166495e-02,8.571273e-17,1.030125e-01,0.000000e+00
MSFT,0.05,0.000000e+00,1.535486e-01,0.000000e+00,2.195523e-01


## Check Portfolio Weights

In [111]:
strategy_weights.sum()

Equal Weight                1.0
Historical Min Vol          1.0
Historical Max Sharpe       1.0
RF Predictive Min Vol       1.0
RF Predictive Max Sharpe    1.0
dtype: float64

In [112]:
strategy_weights.max()

Equal Weight                0.05
Historical Min Vol          0.25
Historical Max Sharpe       0.25
RF Predictive Min Vol       0.25
RF Predictive Max Sharpe    0.25
dtype: float64

## Evaluate Test-Period Performance

In [113]:
portfolio_results = []

for strategy_name in strategy_weights.columns:
    result = evaluate_portfolio(
        strategy_name,
        strategy_weights[strategy_name].values,
        test_returns,
        test_risk_free_rate
    )
    portfolio_results.append(result)

spy_result = evaluate_portfolio(
    "SPY Benchmark",
    np.array([1.0]),
    test_benchmark_returns.to_frame(),
    test_risk_free_rate
)

portfolio_results.append(spy_result)

In [114]:
portfolio_performance = pd.DataFrame(portfolio_results)

portfolio_performance = portfolio_performance.sort_values(
    "sharpe_ratio",
    ascending=False
).reset_index(drop=True)

In [115]:
portfolio_performance

,strategy,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,cumulative_return
0,Historical Max Sharpe,0.266906,0.137115,1.604177,-0.122313,0.668227
1,RF Predictive Max Sharpe,0.265683,0.149006,1.467952,-0.143468,0.658622
2,Historical Min Vol,0.152073,0.076429,1.375432,-0.061527,0.345074
3,Equal Weight,0.173692,0.105403,1.202455,-0.105707,0.396826
4,RF Predictive Min Vol,0.132727,0.075796,1.131689,-0.067824,0.294479
5,SPY Benchmark,0.210936,0.163735,1.001534,-0.187552,0.481125


## Create Daily and Cumulative Return Tables

In [116]:
portfolio_daily_returns = pd.DataFrame(index=test_returns.index)

for strategy_name in strategy_weights.columns:
    portfolio_daily_returns[strategy_name] = test_returns.dot(
        strategy_weights[strategy_name].values
    )

In [117]:
portfolio_daily_returns["SPY Benchmark"] = test_benchmark_returns

In [118]:
portfolio_daily_returns.head()

,Equal Weight,Historical Min Vol,Historical Max Sharpe,RF Predictive Min Vol,RF Predictive Max Sharpe,SPY Benchmark
Date,,,,,,
2024-01-02,0.002311,0.001318,-0.005171,-0.000830,-0.006888,-0.005596
2024-01-03,-0.001278,0.000074,0.006678,0.001061,0.006581,-0.008167
2024-01-04,-0.002874,-0.003578,-0.005485,-0.003901,-0.005532,-0.003221
2024-01-05,0.000533,-0.001106,0.000253,-0.002084,0.000583,0.001370
2024-01-08,0.008257,0.002181,0.009268,0.004608,0.011805,0.014276


In [119]:
portfolio_cumulative_returns = (1 + portfolio_daily_returns).cumprod() - 1

portfolio_cumulative_returns.tail()

,Equal Weight,Historical Min Vol,Historical Max Sharpe,RF Predictive Min Vol,RF Predictive Max Sharpe,SPY Benchmark
Date,,,,,,
2025-12-23,0.392330,0.350913,0.678365,0.295200,0.665013,0.483173
2025-12-24,0.398580,0.354939,0.682153,0.299559,0.669641,0.488391
2025-12-26,0.399609,0.357627,0.686303,0.299706,0.672224,0.488240
2025-12-29,0.396090,0.345445,0.668443,0.295444,0.659145,0.482936
2025-12-30,0.396826,0.345074,0.668227,0.294479,0.658622,0.481125


# Save Portfolio Outputs

In [120]:
strategy_weights.to_csv(PORTFOLIO_OUTPUT_PATH / "portfolio_strategy_weights.csv")
portfolio_performance.to_csv(PORTFOLIO_OUTPUT_PATH / "portfolio_performance_metrics.csv", index=False)
portfolio_daily_returns.to_csv(PORTFOLIO_OUTPUT_PATH / "portfolio_daily_returns.csv")
portfolio_cumulative_returns.to_csv(PORTFOLIO_OUTPUT_PATH / "portfolio_cumulative_returns.csv")

print('Done')

Done


# Conclusion

## Interpretation

The best overall strategy in the test period was the Historical Maximum Sharpe portfolio. It had the highest Sharpe ratio at about 1.60, the highest cumulative return at about 66.8%, and a strong annualized return of about 26.7%. This means that, during the 2024+ test period, the historical return and covariance estimates produced the strongest risk-adjusted portfolio.

The Random Forest Predictive Maximum Sharpe portfolio was the second-best strategy. It had a Sharpe ratio of about 1.47 and a cumulative return of about 65.9%, which was very close to the Historical Maximum Sharpe portfolio. This shows that the Random Forest predicted volatility was useful for portfolio construction, especially for building a high-return strategy, but it did not beat the historical maximum Sharpe portfolio in this test.

The minimum-volatility portfolios had much lower risk. The RF Predictive Minimum Volatility portfolio had the lowest annualized volatility, while the Historical Minimum Volatility portfolio had the smallest max drawdown and a better Sharpe ratio. This means these strategies are better suited for a conservative investor who cares more about reducing losses and volatility than maximizing return.

Compared with the SPY benchmark, most optimized portfolios had better Sharpe ratios and smaller drawdowns. SPY had a strong cumulative return, but it also had the highest volatility and the worst max drawdown. This supports the idea that portfolio optimization can improve risk-adjusted performance compared with simply holding the market benchmark.

Overall, the portfolio optimization step worked. The results show that different strategies match different risk profiles: minimum-volatility portfolios are better for conservative risk preferences, while maximum-Sharpe portfolios are better for investors seeking stronger risk-adjusted returns. The Random Forest predictions added useful predictive risk information, but the historical maximum Sharpe portfolio performed best in this version of the notebook.